# ASSIGNMENT GenAI –  AI Resume Screening System with Tracing



In [1]:

!pip install langchain langchain-groq langsmith groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00


## imports


In [2]:
import os

os.environ["GROQ_API_KEY"]          = "your-groq-api-key-here"
os.environ["LANGCHAIN_API_KEY"]     = "your-langsmith-api-key-here"
os.environ["LANGCHAIN_PROJECT"]     = "ai-resume-screening"
os.environ["LANGCHAIN_TRACING_V2"]  = "true"

print("Environment variables set.")
print(f"LangSmith Tracing: {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"LangSmith Project: {os.environ.get('LANGCHAIN_PROJECT')}")

Environment variables set.
LangSmith Tracing: true
LangSmith Project: ai-resume-screening


##  Inputs

In [3]:
job_description = """
Job Title: Data Scientist

We are looking for a skilled Data Scientist to join our team.

Required Skills:
- Python (Pandas, NumPy, Scikit-learn)
- Machine Learning (supervised and unsupervised)
- Deep Learning (TensorFlow or PyTorch)
- SQL and data wrangling
- Data visualization (Matplotlib, Seaborn, or Tableau)
- Experience with cloud platforms (AWS/GCP/Azure)
- Strong communication and problem-solving skills

Experience Required: 2+ years in a data science or ML role

Nice to Have:
- NLP / LLM experience
- MLflow or experiment tracking tools
- Docker / Kubernetes
"""

#Strong candidate
resume_strong = """
Name: Aditi Sharma
Email: aditi.sharma@email.com

Summary:
Data Scientist with 4 years of experience building ML models and deploying
AI solutions in production environments.

Skills:
- Python (Pandas, NumPy, Scikit-learn, TensorFlow, PyTorch)
- Machine Learning: Regression, Classification, Clustering, Random Forest, XGBoost
- Deep Learning: CNNs, RNNs, Transformers
- SQL, PostgreSQL, BigQuery
- Data Visualization: Matplotlib, Seaborn, Tableau
- Cloud: AWS (S3, SageMaker, EC2), GCP
- NLP, LangChain, OpenAI APIs
- MLflow, Docker

Experience:
- Data Scientist at TechCorp (2021–Present, 3 years)
  * Built customer churn prediction model (92% accuracy)
  * Deployed NLP pipeline for sentiment analysis on AWS
  * Led a team of 2 junior data scientists

- Junior Data Analyst at DataWorks (2020–2021, 1 year)
  * Performed EDA and built dashboards in Tableau

Education:
- B.Tech in Computer Science, IIT Bombay (2020)
"""

#Average Candidate
resume_average = """
Name: Rahul Mehta
Email: rahul.mehta@email.com

Summary:
Aspiring Data Scientist with 1.5 years of experience in data analysis
and some exposure to machine learning projects.

Skills:
- Python (Pandas, NumPy, Scikit-learn)
- Machine Learning: Linear Regression, Decision Trees
- SQL (basic queries)
- Data Visualization: Matplotlib
- Some experience with Jupyter Notebook

Experience:
- Data Analyst at Analytics Pvt Ltd (2023–Present, 1.5 years)
  * Cleaned datasets and created Excel reports
  * Built a basic sales forecasting model using Scikit-learn
  * Wrote SQL queries for data extraction

Education:
- B.Sc. in Statistics, Pune University (2022)

Courses:
- Coursera Machine Learning Specialization (Andrew Ng)
"""

#Weak candidate
resume_weak = """
Name: Sneha Patil
Email: sneha.patil@email.com

Summary:
Recent graduate interested in data and technology. Looking to start a
career in the tech industry.

Skills:
- Microsoft Excel
- Basic Python (learned from YouTube tutorials)
- PowerPoint presentations
- Internet research

Experience:
- Intern at Local NGO (2023, 2 months)
  * Prepared Excel sheets for donation tracking
  * Assisted with social media posts

Education:
- B.Com in Accounting, Pune University (2023)
"""

#Store all resumes in a list for easy iteration
resumes = [
    {"label": "Strong",  "name": "Aditi Sharma",  "text": resume_strong},
    {"label": "Average", "name": "Rahul Mehta",   "text": resume_average},
    {"label": "Weak",    "name": "Sneha Patil",   "text": resume_weak},
]

print("Job description and 3 resumes loaded.")

Job description and 3 resumes loaded.


## Prompts Module

In [5]:
from langchain_core.prompts import PromptTemplate

#Step 1 — Skill Extraction Prompt
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a resume parser. Extract ONLY the information explicitly stated in the resume.
Do NOT assume or infer skills that are not mentioned.

Resume:
{resume}

Extract and return the following in plain text:
Skills: <comma-separated list>
Experience: <total years and roles>
Tools: <specific tools and technologies mentioned>

Important: Only include items clearly stated in the resume above.
"""
)

#Step 2 — Matching Prompt
matching_prompt = PromptTemplate(
    input_variables=["extracted_info", "job_description"],
    template="""
You are a technical recruiter. Compare the candidate's profile against the job requirements.

Candidate Profile:
{extracted_info}

Job Description:
{job_description}

Provide a comparison in plain text:
Matched Skills: <skills present in both the candidate profile and job requirements>
Missing Skills: <skills required by the job but absent from the candidate profile>
Experience Match: <does the candidate meet the experience requirement? Yes / Partial / No>

Important: Base your comparison strictly on the candidate profile provided.
"""
)

#Step 3 — Scoring Prompt
scoring_prompt = PromptTemplate(
    input_variables=["match_result", "job_description"],
    template="""
You are a hiring system. Assign a fit score from 0 to 100 based on the match result.

Scoring guide:
- 80–100: Strong fit (most required skills matched, experience met)
- 50–79:  Moderate fit (some required skills matched, partial experience)
- 0–49:   Weak fit (few or no required skills matched)

Match Result:
{match_result}

Job Description:
{job_description}

Respond with ONLY:
Score: <number between 0 and 100>
"""
)

#Step 4 — Explanation Prompt

explanation_prompt = PromptTemplate(
    input_variables=["score", "match_result", "extracted_info"],
    template="""
You are an AI hiring assistant providing transparent feedback to a recruiter.

Candidate Profile:
{extracted_info}

Match Result:
{match_result}

Assigned Score: {score}

Write a short explanation (3–5 sentences) justifying the score.
Mention specific matched and missing skills.
Do NOT mention skills or facts not present in the candidate profile or match result above.
"""
)

print("All prompt templates created.")

All prompt templates created.


## Chains Module

In [9]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

parser = StrOutputParser()

extraction_chain  = extraction_prompt  | llm | parser
matching_chain    = matching_prompt    | llm | parser
scoring_chain     = scoring_prompt     | llm | parser
explanation_chain = explanation_prompt | llm | parser

print("All LCEL chains created.")

All LCEL chains created.


## Main Pipeline

In [10]:
def screen_resume(resume_text: str, job_desc: str, candidate_label: str) -> dict:
    """
    Full pipeline: Extract → Match → Score → Explain.
    Each step uses .invoke() as required.
    All steps are traced automatically by LangSmith.

    Args:
        resume_text:      Raw resume text
        job_desc:         Job description text
        candidate_label:  Label for display (Strong / Average / Weak)

    Returns:
        dict with extracted_info, match_result, score, explanation
    """
    print(f"\n{'='*60}")
    print(f"Processing: {candidate_label} Candidate")
    print(f"{'='*60}")

    #Step 1 — Skill Extraction
    print("[Step 1] Extracting skills...")
    extracted_info = extraction_chain.invoke({"resume": resume_text})
    print(extracted_info)

    #Step 2 — Matching
    print("\n[Step 2] Matching against job description...")
    match_result = matching_chain.invoke({
        "extracted_info": extracted_info,
        "job_description": job_desc
    })
    print(match_result)

    #Step 3 — Scoring
    print("\n[Step 3] Scoring...")
    score_output = scoring_chain.invoke({
        "match_result": match_result,
        "job_description": job_desc
    })
    print(score_output)

    #Step 4 — Explanation
    print("\n[Step 4] Generating explanation...")
    explanation = explanation_chain.invoke({
        "score": score_output,
        "match_result": match_result,
        "extracted_info": extracted_info
    })
    print(explanation)

    return {
        "candidate":      candidate_label,
        "extracted_info": extracted_info,
        "match_result":   match_result,
        "score":          score_output,
        "explanation":    explanation
    }

print("Pipeline function defined.")

Pipeline function defined.


##  Running the Pipeline on All 3 Resumes


In [11]:
#Store results for the summary table
all_results = []

for resume_entry in resumes:
    result = screen_resume(
        resume_text=resume_entry["text"],
        job_desc=job_description,
        candidate_label=f"{resume_entry['label']} ({resume_entry['name']})"
    )
    all_results.append(result)

print("\n All 3 resumes processed. Check LangSmith for traces.")


Processing: Strong (Aditi Sharma) Candidate
[Step 1] Extracting skills...
Skills: Python, Machine Learning, Deep Learning, SQL, PostgreSQL, BigQuery, Data Visualization, Cloud, NLP, LangChain, OpenAI APIs, MLflow, Docker
Experience: 4 years, Data Scientist, Junior Data Analyst
Tools: Pandas, NumPy, Scikit-learn, TensorFlow, PyTorch, Matplotlib, Seaborn, Tableau, AWS, S3, SageMaker, EC2, GCP, XGBoost, CNNs, RNNs, Transformers, MLflow, Docker

[Step 2] Matching against job description...
Matched Skills: Python, Machine Learning, Deep Learning, SQL, Data Visualization, Cloud, Pandas, NumPy, Scikit-learn, TensorFlow, PyTorch, Matplotlib, Seaborn, Tableau, AWS, GCP, MLflow, Docker
Missing Skills: Azure, Kubernetes, specific mention of supervised and unsupervised machine learning, specific mention of strong communication and problem-solving skills
Experience Match: Yes

[Step 3] Scoring...
Score: 85

[Step 4] Generating explanation...
The candidate has been assigned a score of 85 due to the

##  Summary Table

In [12]:
import re

print(f"\n{'CANDIDATE':<35} {'SCORE':>6}")
print("-" * 45)

for r in all_results:
    match = re.search(r'\d+', r['score'])
    score_val = match.group() if match else "N/A"
    print(f"{r['candidate']:<35} {score_val:>6}")

print("-" * 45)
print("\nNote: Scores are out of 100.")


CANDIDATE                            SCORE
---------------------------------------------
Strong (Aditi Sharma)                   85
Average (Rahul Mehta)                   60
Weak (Sneha Patil)                      10
---------------------------------------------

Note: Scores are out of 100.


In [13]:

print("Running debug test with empty resume...\n")

debug_resume = "Name: Test User"  #intentionally minimal — no skills, no experience

try:
    debug_result = screen_resume(
        resume_text=debug_resume,
        job_desc=job_description,
        candidate_label="Debug (Incomplete Resume)"
    )
    print("\n[DEBUG] Output generated — check LangSmith trace for this run.")
    print("[DEBUG] Expected: low score and explanation noting missing information.")
except Exception as e:
    print(f"[DEBUG] Error captured: {e}")
    print("[DEBUG] This error is now visible in LangSmith for debugging.")

Running debug test with empty resume...


Processing: Debug (Incomplete Resume) Candidate
[Step 1] Extracting skills...
Skills: 
Experience: 
Tools: 

(Note: Since the provided resume only contains the name "Test User" and no other information, the fields for skills, experience, and tools are left blank.)

[Step 2] Matching against job description...
Matched Skills: None
Missing Skills: Python (Pandas, NumPy, Scikit-learn), Machine Learning (supervised and unsupervised), Deep Learning (TensorFlow or PyTorch), SQL and data wrangling, Data visualization (Matplotlib, Seaborn, or Tableau), Experience with cloud platforms (AWS/GCP/Azure), Strong communication and problem-solving skills, NLP / LLM experience, MLflow or experiment tracking tools, Docker / Kubernetes
Experience Match: No 

Note: Since the candidate profile does not contain any information about skills or experience, it is not possible to find any matched skills, and all required skills are considered missing. The experience ma

In [20]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [22]:
import nbformat, shutil

path = "/drive/MyDrive/Colab Notebooks/GenAI_Task_3_AI_ResumeScreeningSystem.ipynb"

shutil.copy(path, path.replace(".ipynb", "_backup.ipynb"))

with open(path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

nb.metadata.pop("widgets", None)

for cell in nb.cells:
    cell.get("metadata", {}).pop("widgets", None)

with open(path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Done")

Done
